In [ ]:
# Cell 1: Install required libraries
# %pip installs into this notebook's active kernel (unlike a bare !pip).
%pip install -q --upgrade pip
%pip install -q pandas
%pip install -q requests beautifulsoup4 chromadb kagglehub google-genai python-dotenv

# The official MCP SDK requires Python 3.10 or newer.
import sys
if sys.version_info < (3, 10):
    raise RuntimeError(
        f"MCP requires Python 3.10+, but this kernel is Python {sys.version.split()[0]}. "
        "Select a Python 3.10+ Jupyter kernel, then rerun this cell. "
        "Pandas and the other dependencies have already been installed."
    )
%pip install -q "mcp[cli]"

In [ ]:
# Cell 2: Data Ingestion & Script Indexer
import os
import re
import sqlite3
import urllib.parse
import pandas as pd
import requests
from bs4 import BeautifulSoup
import chromadb
import kagglehub

# Download latest version
dataset_dir = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")

print("📥 Step 1: Downloading IMDb Top 1000 dataset...")
# Mirror of the Kaggle IMDb Top 1000 Dataset
csv_path = os.path.join(dataset_dir, "imdb_top_1000.csv")
df = pd.read_csv(csv_path)

# Save metadata to SQLite
conn = sqlite3.connect("movies_metadata.db")
df.to_sql("movies", conn, if_exists="replace", index=False)
conn.close()
print(f"✅ Saved {len(df)} movies to movies_metadata.db")

# Initialize ChromaDB Vector Store
chroma_client = chromadb.PersistentClient(path="./screenplay_db")
dialogue_collection = chroma_client.get_or_create_collection(name="screenplay_dialogues")

def fetch_imsdb_script(movie_title):
    """Attempt to scrape script text from IMSDb."""
    headers = {"User-Agent": "Mozilla/5.0"}
    # IMSDb URL conventions: "Pulp Fiction" -> "Pulp-Fiction"
    slug = urllib.parse.quote(movie_title.strip().replace(" ", "-"))
    url = f"https://imsdb.com/scripts/{slug}.html"

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.content, "html.parser")
        scrtext = soup.find("td", class_="scrtext")
        if scrtext:
            return scrtext.get_text()
    except Exception:
        return None
    return None

def chunk_script(script_text, max_chunk_chars=1200):
    """Splits raw screenplay text into scene-sized chunks."""
    # Split on scene headers (INT. or EXT.)
    raw_chunks = re.split(r'(?=\n(?:INT\.|EXT\.))', script_text)
    clean_chunks = []
    for c in raw_chunks:
        text = c.strip()
        if len(text) > 80:  # Ignore trivial noise
            clean_chunks.append(text[:max_chunk_chars])
    return clean_chunks

print("\n🎬 Step 2: Indexing sample scripts from IMSDb into ChromaDB...")
# We index top 10 available scripts by default to finish quickly; increase as needed
indexed_count = 0
target_movies = df['Series_Title'].tolist()

for title in target_movies:
    if indexed_count >= 10:  # Adjust to index more scripts
        break

    script_text = fetch_imsdb_script(title)
    if not script_text:
        continue

    chunks = chunk_script(script_text)
    if not chunks:
        continue

    docs = []
    metas = []
    ids = []
    for idx, chunk in enumerate(chunks[:25]): # index up to 25 scenes per script
        docs.append(chunk)
        metas.append({"movie_title": title, "scene_idx": idx})
        ids.append(f"{title}_{idx}")

    dialogue_collection.upsert(documents=docs, metadatas=metas, ids=ids)
    print(f"  ✓ Indexed: {title} ({len(docs)} scenes)")
    indexed_count += 1

print(f"\n✅ Finished! Indexed {indexed_count} full scripts into vector store.")

In [ ]:
%%writefile server.py
import sqlite3
import chromadb
from mcp.server import MCPServer

mcp = MCPServer("ScreenplayReferenceServer")

# Connect to database and vector collection
chroma_client = chromadb.PersistentClient(path="./screenplay_db")
dialogue_collection = chroma_client.get_or_create_collection(name="screenplay_dialogues")
SQLITE_DB = "movies_metadata.db"

@mcp.tool()
def search_top_movies(genre: str = None, min_rating: float = 8.0, limit: int = 5) -> str:
    """Search IMDb Top 1000 movies by genre and rating."""
    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()
    query = "SELECT Series_Title, Released_Year, IMDB_Rating, Genre, Director, Overview FROM movies WHERE IMDB_Rating >= ?"
    params = [min_rating]
    if genre:
        query += " AND Genre LIKE ?"
        params.append(f"%{genre}%")
    query += " ORDER BY IMDB_Rating DESC LIMIT ?"
    params.append(limit)

    cursor.execute(query, params)
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No movies found matching the criteria."

    results = []
    for r in rows:
        results.append(f"Title: {r[0]} ({r[1]})\nRating: {r[2]} | Genre: {r[3]}\nDirector: {r[4]}\nLogline: {r[5]}")
    return "\n\n".join(results)

@mcp.tool()
def search_dialogue_and_scenes(query: str, movie_title: str = None, limit: int = 2) -> str:
    """Semantic vector search across movie scenes and dialogues."""
    if dialogue_collection.count() == 0:
        return "No screenplay scenes are indexed yet. Run Cell 2 first."
    limit = max(1, min(limit, dialogue_collection.count()))
    where_filter = {"movie_title": movie_title} if movie_title else None
    query_args = {"query_texts": [query], "n_results": limit}
    if where_filter:
        query_args["where"] = where_filter
    results = dialogue_collection.query(**query_args)

    if not results or not results["documents"][0]:
        return "No matching dialogues or scenes found in the indexed scripts."

    output = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        output.append(f"=== Film: {meta['movie_title']} (Scene {meta['scene_idx']}) ===\n{doc}")
    return "\n\n".join(output)

if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing server.py


In [ ]:
# Cell 4: Scriptwriter Agent Client
import os
import importlib
from getpass import getpass
from dotenv import load_dotenv
from google import genai
from mcp import Client
import server  # MCP server generated in Cell 3
importlib.reload(server)  # Pick up changes when Cell 3 is rerun

# Read the key from the environment, Colab Secrets, or a hidden prompt.
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
  try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
  except Exception:  # Not in Colab, or the named secret does not exist
    pass
if not api_key:
  api_key = getpass("Enter your Gemini API key: ").strip()
if not api_key:
  raise ValueError("A GEMINI_API_KEY is required.")

# Initialize the Gemini Client
gemini_client = genai.Client(api_key=api_key)


def tool_result_text(result):
  """Extract text returned by an MCP tool call."""
  return "\n".join(
      block.text for block in result.content
      if getattr(block, "type", None) == "text"
  )


async def run_writer_assistant(prompt_query: str):
  print(f'🤖 User Query: "{prompt_query}"\n')

  # Call both registered tools through an in-process MCP client.
  print("🔍 Agent: Searching screenplay knowledge base...")
  user_genre = input("Enter your movie's genre: ")
  async with Client(server.mcp) as mcp_client:
    references_result = await mcp_client.call_tool(
        "search_dialogue_and_scenes",
        {"query": prompt_query, "limit": 2},
    )
    movies_result = await mcp_client.call_tool(
        "search_top_movies",
        {"genre": user_genre, "limit": 2},
    )
  raw_references = tool_result_text(references_result)
  movie_context = tool_result_text(movies_result)

  # Step 3: Synthesize advice using Gemini
  system_instruction = (
      "You are an expert Hollywood Screenwriting Consultant. "
      "Analyze the screenplay excerpts provided from the IMDb Top 1000 database.\n"
      "Help the writer pick out a popular quote from the movie that can be referenced by the writer in the script they provided.\n"
  )

  user_message = f"""
    Writer's goal/scene idea:
    "{prompt_query}"

    Screenplay References retrieved:
    {raw_references}

    Top Film Suggestions:
    {movie_context}
    """

  # Send prompt to Gemini
  response = gemini_client.models.generate_content(
      model="gemini-3.7-flash",
      contents=f"{system_instruction}\n\n{user_message}",
  )

  print("\n" + "=" * 60)
  print("🎬 SCREENPLAY CONSULTANT ADVICE:")
  print("=" * 60)
  print(response.text)


# Prompt the user for input dynamically
user_prompt = input("Enter your scene idea: ")

# Run the assistant with the user's input
if user_prompt.strip():
  await run_writer_assistant(user_prompt)
else:
  print("No prompt provided. Please enter a valid scene idea.")

Enter your scene idea: A character tries to control his dreams
🤖 User Query: "A character tries to control his dreams"

🔍 Agent: Searching screenplay knowledge base...
Enter your movie's genre: Drama

🎬 SCREENPLAY CONSULTANT ADVICE:
Here is a screenwriting consultation analysis based on your scene concept and the referenced film, Christopher Nolan’s ***Inception***.

---

### **Top Quote Recommendations from *Inception***

#### **1. The Pop-Culture Standout (Best for Wit & Escalation)**
> **“You mustn’t be afraid to dream a little bigger, darling.”**  
> *— Eames (Tom Hardy)*

* **Why it works:** This is the most iconic, widely quoted line from *Inception*. In the film, Eames says this right before pulling out a grenade launcher to overpower a dream projection. 
* **How to integrate it:** If your character is struggling to conjure or alter something in their dream (e.g., trying to imagine a locked door opening or summoning a small tool) and another character (or their own inner monolog